# 图文一致性分析 Demo

本 Notebook 演示如何使用图文相似度分析模型来分析文本与图像的一致性。

## 1. 设置 HuggingFace 镜像源（国内加速）

In [ ]:
import os

# 设置 HuggingFace 镜像源（国内加速）
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

# 设置模型缓存目录
os.environ['HF_HOME'] = './models/huggingface'

# 设置下载超时时间（秒）
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '300'

print("已设置 HuggingFace 镜像源: https://hf-mirror.com")
print(f"模型缓存目录: {os.environ['HF_HOME']}")

## 2. 安装依赖

In [ ]:
# 安装项目依赖
import subprocess
import sys

packages = [
    "PyYAML>=6.0",
    "requests>=2.28.0",
    "numpy>=1.24.0",
    "pillow>=9.0.0",
    "hanlp>=2.0.0",
    "FlagEmbedding>=1.2.0",
    "torch>=2.0.0",
    "transformers>=4.40.0",
    "qwen-vl-utils>=0.0.2",
    "accelerate>=0.25.0"
]

print("正在安装依赖...")
subprocess.run([sys.executable, "-m", "pip", "install"] + packages + ["-q"])
print("依赖安装完成!")

## 3. 加载模块

In [ ]:
import sys
import json
import warnings
warnings.filterwarnings('ignore')

# 添加项目路径
sys.path.insert(0, os.path.abspath('.'))

from src.text_processor import TextProcessor
from src.image_processor import ImageProcessor
from src.vectorizer import Vectorizer
from src.similarity import SimilarityCalculator

## 4. 配置

In [ ]:
# ==================== 配置区域 ====================

# AutoDL 社区镜像预装模型路径
LOCAL_MODEL_PATH = "/root/models/Qwen2.5-VL-7B-Instruct"

# BGE-M3 模型（使用镜像源会自动从 hf-mirror.com 下载）
BGE_MODEL = "BAAI/bge-m3"

# 选择 provider: "autodl", "ollama", "huggingface", "openrouter"
PROVIDER = "autodl"  # 使用 AutoDL 社区镜像

print(f"使用 Provider: {PROVIDER}")
print(f"模型路径: {LOCAL_MODEL_PATH}")
print(f"BGE模型: {BGE_MODEL}")

## 5. 初始化模型

In [ ]:
print("=" * 50)
print("初始化模型...")
print("=" * 50)

# 文本处理器 (HanLP)
print("\n[1/3] 加载 HanLP...")
text_processor = TextProcessor()
text_processor.load()
print("✓ HanLP 加载完成")

# 图像处理器 (Qwen2.5-VL)
print("\n[2/3] 加载 Qwen2.5-VL...")
if PROVIDER == "autodl":
    image_processor = ImageProcessor(
        provider="autodl",
        local_model_path=LOCAL_MODEL_PATH
    )
else:
    image_processor = ImageProcessor(
        provider=PROVIDER,
        model="qwen2.5-vl:7b-instruct",
        base_url="http://localhost:11434"
    )
image_processor.load()
print("✓ Qwen2.5-VL 加载完成")

# 向量化器 (BGE-M3)
print("\n[3/3] 加载 BGE-M3...")
print("注意：首次加载会从 hf-mirror.com 下载模型，请耐心等待...")
vectorizer = Vectorizer(model_name=BGE_MODEL)
vectorizer.load()
print("✓ BGE-M3 加载完成")

# 相似度计算器
similarity_calc = SimilarityCalculator()

print("\n" + "=" * 50)
print("所有模型加载完成！")
print("=" * 50)

## 6. 准备输入数据

In [ ]:
# ==================== 输入数据 ====================
# 请修改以下数据为你需要分析的图文

input_data = {
    "id": "001",
    "text": "这家酒店位于繁华的市中心，装修豪华，服务周到，房间干净整洁",
    "image_path": "https://example.com/hotel.jpg"  # 可以是 URL 或本地路径
}

print("输入数据:")
print(json.dumps(input_data, ensure_ascii=False, indent=2))

## 7. 执行分析

In [ ]:
import time

print("=" * 50)
print("开始分析...")
print("=" * 50)

start_time = time.time()

# 1. 文本处理 - 抽取名词和形容词
print("\n[1/3] 文本处理...")
text_result = text_processor.process(input_data['text'])
print(f"  文本名词: {text_result['nouns']}")
print(f"  文本形容词: {text_result['adjectives']}")

# 2. 图像处理 - 提取图像中的名词和形容词
print("\n[2/3] 图像处理...")
image_result = image_processor.process(input_data['image_path'])
print(f"  图像名词: {image_result['nouns']}")
print(f"  图像形容词: {image_result['adjectives']}")

# 3. 相似度计算
print("\n[3/3] 相似度计算...")
similarity_result = similarity_calc.compute_similarities(
    text_nouns=text_result['noun_string'],
    text_adjectives=text_result['adjective_string'],
    image_nouns=image_result['noun_string'],
    image_adjectives=image_result['adjective_string'],
    vectorizer=vectorizer
)

processing_time = time.time() - start_time

print("\n" + "=" * 50)
print("分析完成!")
print("=" * 50)

## 8. 结果展示

In [ ]:
# 构建结果
result = {
    "id": input_data['id'],
    "text": input_data['text'],
    "image_path": input_data['image_path'],
    "success": True,
    "noun_similarity": similarity_result['noun_similarity'],
    "adjective_similarity": similarity_result['adjective_similarity'],
    "average_similarity": similarity_result['average_similarity'],
    "details": {
        "text_nouns": text_result['nouns'],
        "text_adjectives": text_result['adjectives'],
        "image_nouns": image_result['nouns'],
        "image_adjectives": image_result['adjectives']
    },
    "processing_time": round(processing_time, 2)
}

# 打印结果
print("=" * 60)
print("📊 图文一致性分析结果")
print("=" * 60)

print(f"\n📌 ID: {result['id']}")
print(f"📝 文本: {result['text']}")
print(f"🖼️  图像: {result['image_path']}")

print("\n" + "-" * 60)
print("📈 相似度评分:")
print(f"   🔤 名词相似度:     {result['noun_similarity']:.4f}")
print(f"   📖 形容词相似度:   {result['adjective_similarity']:.4f}")
print(f"   📊 平均相似度:     {result['average_similarity']:.4f}")

print("\n" + "-" * 60)
print("📋 详细分析:")
print(f"   文本名词:     {result['details']['text_nouns']}")
print(f"   文本形容词:   {result['details']['text_adjectives']}")
print(f"   图像名词:     {result['details']['image_nouns']}")
print(f"   图像形容词:   {result['details']['image_adjectives']}")

print("\n" + "-" * 60)
print(f"⏱️  处理时间: {result['processing_time']} 秒")

print("\n" + "=" * 60)

## 9. 保存结果（可选）

In [ ]:
# 保存结果到文件
output_file = "data/output/demo_result.json"
os.makedirs(os.path.dirname(output_file), exist_ok=True)

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"结果已保存至: {output_file}")